# Lesson 18: Securing AI Agents wit Cryptographic Receipts

## Hands-on Notebook

Dis notebook go waka you through four exercises:

1. **Sign your first receipt** for one agent tool call and verify am.
2. **Tamper wit di receipt** and watch as verification fail.
3. **Build three-receipt chain** and confirm say chain dey intact.
4. **Wrap Microsoft Agent Framework tool call** so every action go release receipt.

All cryptographic primitives dey imported from well-maintained libraries (`pynacl` for Ed25519, `jcs` for RFC 8785 canonical JSON, `hashlib` from the Python standard library for SHA-256). Di receipt logic na plain Python wey you fit read and modify.

Run di cells in order. Every section short and self-contained.


## Setup

Install di two dependencies. Both get permissive licenses (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

These two helpers dey handle base64url encoding (without padding) and SHA-256 hashing of any kain object. Dem dey keep the rest of the notebook focused on the receipt logic itself.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Sign your first receipt

Imagine say our agent for **Contoso Travel** just check flight from Sydney to Los Angeles for one customer. We want to record this tool call as signed receipt so say future auditor fit verify am without to trust us.

### Step 1.1: Generate a signing key

For production, the agent signing key for inside hardware security module (HSM), Azure Key Vault, or another protected store. For this lesson we go generate fresh key for memory.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Build di receipt payload

Di payload get everytin we want di receipt to confirm: who act, which tool, wit which arguments, wetin come back, under which policy, and wen. We dey hash di arguments and result instead make we put dem inline so dat di receipt no go leak sensitive content.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: Sign and assemble di receipt

Three steps:

1. Make di payload canonical wit JCS so two implementations wey dey produce di same logical receipt go produce byte-identical bytes.
2. Hash di canonical bytes wit SHA-256.
3. Sign di hash wit di Ed25519 private key.

Den di signature go attach to di original payload to produce di final receipt.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: Verify di receipt

Verification dey reverse di process. We go remove di signature, recompute di canonical hash, and check di signature against di public key wey dey inside di receipt.

Auditor wey dey do dis verification no need anything from us again except di receipt itself. No service wey e need call, no key directory wey e need check, no trust wey e need.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

You for see `Receipt is valid: True`. Di agent don produce im first cryptographically signed audit record.


## Section 2: Tamper wit di receipt

Di whole point of receipts na say dem dey tamper-evident. Make we prove am.

We go modify exactly one character for di receipt and watch verification fail.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Wetin just happen?

When we change `policy_id`, di canonical bytes change. Di SHA-256 hash of dem bytes change. Di signature (wey bin dey for di original hash) no gree match di new hash again. Verification correctly dey return `False`.

No possible way to change any field for di receipt and still make am verify, unless di attacker get di private key. As long as di private key dey for key vault and di public key don publish, tampering no fit hide.

Try am yourself: change di `tool_name` or `agent_id` or `timestamp` for di cell wey dey above and run am again. Every change go give you invalid receipt.


## Section 3: Chain receipts together

One single receipt dey protect one action. Most agents dey do plenti actions. To make the whole sequence tamper-evident, we dey join each receipt to the previous one by putting the previous receipt's hash inside the new receipt's payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

If pesin comot or change the order of receipt, the chain go break for that exact point. When you try check any later receipt, e go fail becos the `previous_receipt_hash` no go match the correct hash wey suppose dey for e predecessor.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Now break di chain by tamper wit di middle receipt and re-verify. Di tampered receipt no pass im signature check, AND di next receipt no pass im chain-link check (because im `previous_receipt_hash` no dey match di modified middle receipt's hash again).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 still dey verify (e no get any modification and e no get any predecessor wey e go depend on). Receipt 1 no pass im signature check because we don change `tool_args_hash`. Receipt 2 no pass im chain-link check because im `previous_receipt_hash` bin compute based on the original (wey dem don modify now) receipt 1.

Even if attacker sign the modified receipt 1 again (wey dem no fit do without the private key), the chain-link mismatch wey dey for receipt 2 go still show say dem don tamper am. To hide the change, the attacker go need sign every receipt from the modification point forward, and to do dat, dem go need the private key.


## Section 4: Wrap an agent tool call with receipt signing

For real deployment, you no go want say every agent author go dey remember to call `make_receipt`. You go want make receipt signing automatic for every tool invocation.

Na dis one be the simplest pattern: wrapper class wey go take any callable tool function come return version wey dey emit receipt. Dis one fit work with any agent framework, including Microsoft Agent Framework (`agent_framework.azure`).

If you never get Azure AI Foundry project set up, the local mock wey dey below still dey show dis pattern well.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrating wit di Microsoft Agent Framework

Di `ReceiptedTool` wrapper wey dey above no dey depend on any framework. To use am inside one agent wey dem build wit di Microsoft Agent Framework, register di wrapped function as tool. One sketch (wey you for change di mock wit real Azure AI Foundry tool registration):

```python
# Pseudocode wey dey show how integration be.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="You be Contoso Travel agent ...",
#     tools=[receipted_lookup],   # na di wrapped tool, no be di raw function
# )
# response = agent.run("Find flights from Sydney go Los Angeles for June.")
#
# # After di run finish, every tool call wey di agent do get signed receipt:
# audit_chain = receipted_lookup.receipts
```

Di agent framework no need sabi anything about receipts. Receipt signing na wrap around di tool, e no join inside di framework. Dis na how you for add provenance to existing agent code without to rewrite di agent.


## Recap and stretch challenge

You don:

- Generate one Ed25519 key pair.
- Build and sign one receipt for one agent tool call.
- Verify the receipt offline using only di public key.
- Tamper with one receipt and see say verification fail.
- Build one hash-chained sequence of three receipts.
- Tamper with di middle of di chain and see both signature failure and chain-link failure.
- Wrap one tool function wit automatic receipt signing.

**Stretch challenge.** Extend di receipt schema wit one `request_id` field (na UUID for distributed tracing). Update `make_receipt` to include am, and confirm say receipts still verify from beginning to end. Then change di field after signing and confirm say verification fail. Dis one go force you to sabi how every byte for di canonical encoding dey add to di signature.

**Important boundary.** Receipts dey prove three tins and na only three tins: attribution (dis key sign dis content), integrity (di content no change since dem sign am), and ordering (dis receipt follow after dat receipt). Dem no dey prove say di agent action correct, say di policy wey get name for `policy_id` really evaluate, or say di agent follow every rule. Receipts na di foundation. Governance na di system wey you build on top.

Read di lesson README again wit dis boundary for mind. Di most common mistake wey teams dey make wit receipts na to assume "we get receipts" mean say "we dey governed." E no mean so. Receipts dey make agent behaviour auditable. Dem no dey make am correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
